# Pruebas de calidad: ejecutar y verificar las reglas

Comprueba que el motor `dq_validaciones` detecta los problemas y toma la decisión configurada en cada regla: **DETENER**, **CUARENTENA** o **ADVERTIR**.

Con los datos reales todas las reglas dan 100 %, porque Silver ya limpió todo. Por eso este notebook **inyecta defectos** directamente en una copia de las tablas y comprueba la respuesta del motor.

## Cómo funciona

1. Copia las tablas Silver y Gold reales, el catálogo de reglas y la tabla de control al esquema `pruebas_reglas_calidad`. Los datos reales no se tocan.
2. Para cada escenario: inyecta el defecto, ejecuta `dq_validaciones` apuntando a la copia, compara lo obtenido contra lo esperado y deshace el defecto con `RESTORE` de Delta.
3. Muestra una tabla con cada verificación y termina con error si alguna falla.

## Escenarios

| Escenario | Momento | Defecto inyectado | Se espera |
|---|---|---|---|
| CONTROL_SILVER | silver | ninguno | las 19 reglas en CUMPLE |
| CONTROL_GOLD | gold | ninguno | las 3 reglas en CUMPLE |
| DUPLICADOS_SILVER | silver | 3 filas duplicadas en pedidos, líneas, clientes y productos | DQ-001 a DQ-004 en DETENIDO; la task falla |
| ADVERTIR_SILVER | silver | fechas de envío vacías o anteriores al pedido, estado desconocido, precios, descuentos y tasas fuera de rango, ventana de control sin datos | DQ-007, 010, 017, 018, 019, 020 y 021 en ADVERTENCIA; la task sigue |
| TOLERANCIA_SILVER | silver | 0,1 % de pedidos sin fecha de envío y 0,5 % de líneas sin precio | DQ-007 y DQ-019 en CUMPLE: están dentro del umbral |
| CUARENTENA_LINEA_SIN_PEDIDO | silver | 600 pedidos borrados de Silver | DQ-011: las líneas huérfanas pasan a CUARENTENA |
| CUARENTENA_LINEA_SIN_PRODUCTO | silver | 2 productos borrados de Silver | DQ-012: las líneas sin producto pasan a CUARENTENA |
| CUARENTENA_PEDIDO_SIN_CLIENTE | silver | 50 clientes borrados de Silver | DQ-013: los pedidos sin cliente pasan a CUARENTENA |
| DETENER_GOLD | gold | claves nulas en el hecho, pedidos faltantes en Gold, resumen duplicado | DQ-014, 015 y 016 en DETENIDO; la task falla |

**No se prueba DQ-022** (caída de volumen): requiere un Bronze con días de volumen distinto.

## Antes de correr

* Debe existir una corrida completa del pipeline real: se copian sus tablas Silver y Gold.
* Cada aviso de las pruebas llega al **Slack real**. Se reconocen porque el `id_ejecucion` empieza con `PRUEBA-DQ`.
* El motor lee el Bronze real solo para DQ-018; no lo modifica.
* Tarda varios minutos: el motor corre 9 veces.

In [0]:
import uuid

ESQUEMA_PRUEBA = "pruebas_reglas_calidad"
SILVER = f"slv_dataco.{ESQUEMA_PRUEBA}"
GOLD = f"gld_dataco.{ESQUEMA_PRUEBA}"
METADATA = f"metadata_dataco.{ESQUEMA_PRUEBA}"

PEDIDOS = f"{SILVER}.hd_pedidos"
DETALLE = f"{SILVER}.hd_pedido_detalle"
CLIENTES = f"{SILVER}.md_clientes"
PRODUCTOS = f"{SILVER}.md_productos"
HECHO_PEDIDOS = f"{GOLD}.desc_ft_orders"
RESUMEN_PEDIDOS = f"{GOLD}.desc_ft_order_summary"
CONTROL = f"{METADATA}.md_pipeline_control"
CATALOGO_REGLAS = f"{METADATA}.md_dq_reglas"
RESULTADOS = f"{METADATA}.hd_dq_resultados"
CUARENTENA_RESUMEN = f"{METADATA}.hd_cuarentena_resumen"
REGISTRO_EJECUCION = f"{METADATA}.hd_etl_log"

RUTA_MOTOR = "../06_calidad_datos/01_negocio/dq_validaciones"
PREFIJO_PRUEBA = f"PRUEBA-DQ-{uuid.uuid4()}"

# El motor lee la copia de Silver, Gold y metadata. El Bronze real se usa solo para DQ-018.
PARAMETROS_MOTOR = {
    "silver_catalog": "slv_dataco", "silver_schema": ESQUEMA_PRUEBA,
    "gold_catalog": "gld_dataco", "gold_schema": ESQUEMA_PRUEBA,
    "bronze_catalog": "brz_dataco", "bronze_schema": "sharepoint", "bronze_table": "raw_supply_chain",
    "metadata_catalog": "metadata_dataco", "metadata_schema": ESQUEMA_PRUEBA,
}

print(f"Prefijo de la prueba: {PREFIJO_PRUEBA}")

In [0]:
# ============================================================
# Preparar la base sana: copia de las tablas reales
# ------------------------------------------------------------
# Solo se reemplazan tablas dentro del esquema de prueba.
# ============================================================

for esquema in (SILVER, GOLD, METADATA):
    assert esquema.endswith(f".{ESQUEMA_PRUEBA}"), "Solo se tocan esquemas de prueba"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema}")

COPIAS = [
    (PEDIDOS, "slv_dataco.supply_chain.hd_pedidos"),
    (DETALLE, "slv_dataco.supply_chain.hd_pedido_detalle"),
    (CLIENTES, "slv_dataco.supply_chain.md_clientes"),
    (PRODUCTOS, "slv_dataco.supply_chain.md_productos"),
    (HECHO_PEDIDOS, "gld_dataco.supply_chain.desc_ft_orders"),
    (RESUMEN_PEDIDOS, "gld_dataco.supply_chain.desc_ft_order_summary"),
    (CONTROL, "metadata_dataco.supply_chain.md_pipeline_control"),
    (CATALOGO_REGLAS, "metadata_dataco.supply_chain.md_dq_reglas"),
]
for destino, origen in COPIAS:
    spark.sql(f"CREATE OR REPLACE TABLE {destino} AS SELECT * FROM {origen}")
    filas = spark.table(destino).count()
    assert filas > 0, f"{destino} quedó vacía: ejecuta primero el pipeline real"
    print(f"✓ {destino}: {filas:,} filas")

# Tablas que escribe el motor: solo estructura, sin filas.
for tabla in ("hd_dq_resultados", "hd_cuarentena_resumen", "hd_etl_log"):
    destino = f"{METADATA}.{tabla}"
    spark.sql(f"DROP TABLE IF EXISTS {destino}")
    spark.sql(f"CREATE TABLE {destino} LIKE metadata_dataco.supply_chain.{tabla}")
    print(f"✓ {destino} (sin filas)")

In [0]:
# ============================================================
# Funciones de apoyo: snapshot, restauración, motor y verificación
# ============================================================

VERIFICACIONES = []   # (escenario, verificacion, esperado, obtenido, estado)


def verificar(escenario, descripcion, esperado, obtenido):
    estado = "OK" if str(esperado) == str(obtenido) else "FALLA"
    VERIFICACIONES.append((escenario, descripcion, str(esperado), str(obtenido), estado))


def contar(consulta):
    return spark.sql(consulta).first()[0]


def tomar_snapshot(tablas):
    """Guarda la versión actual de cada tabla para poder deshacer el defecto."""
    return {t: spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"] for t in tablas}


def restaurar(snapshot):
    for tabla, version in snapshot.items():
        assert f".{ESQUEMA_PRUEBA}." in tabla, "Solo se restauran tablas de prueba"
        spark.sql(f"RESTORE TABLE {tabla} TO VERSION AS OF {version}")


def identificadores(tabla, columna, condicion, orden, cantidad):
    """Primeras (ASC) o últimas (DESC) claves de una tabla: el defecto es determinista."""
    filas = spark.sql(
        f"SELECT {columna} FROM {tabla} WHERE {condicion} ORDER BY {columna} {orden} LIMIT {cantidad}"
    ).collect()
    return [fila[0] for fila in filas]


def lista(valores):
    return ",".join(str(v) for v in valores)


def reglas_activas(momento):
    return contar(f"SELECT COUNT(*) FROM {CATALOGO_REGLAS} WHERE estado = 'ACTIVA' AND momento_ejecucion = '{momento}'")


def ejecutar_motor(momento, id_ejecucion):
    """Corre dq_validaciones. Devuelve (detuvo, mensaje): detuvo es True si la task falla."""
    parametros = dict(PARAMETROS_MOTOR, momento=momento, id_ejecucion_manual=id_ejecucion)
    try:
        dbutils.notebook.run(RUTA_MOTOR, 3000, parametros)
        return False, ""
    except Exception as error:
        return True, str(error)


def correr_escenario(nombre, momento, tablas_afectadas, inyectar, debe_detener):
    """
    inyectar(id_ejecucion) aplica el defecto y devuelve (esperado, extras):
      esperado: {regla: (resultado, filas_incumplen, filas_aisladas o None)}
      extras:   [(descripcion, valor_esperado, consulta)] que se evalúan antes de restaurar
    Las reglas que no están en 'esperado' deben dar CUMPLE.
    """
    id_ejecucion = f"{PREFIJO_PRUEBA}-{nombre}"
    print(f"▶ {nombre}")
    snapshot = tomar_snapshot(tablas_afectadas)
    try:
        esperado, extras = inyectar(id_ejecucion)
        detuvo, mensaje = ejecutar_motor(momento, id_ejecucion)

        verificar(nombre, "la task se detiene" if debe_detener else "la task sigue sin error", debe_detener, detuvo)
        resultados = {
            fila["regla_id"]: fila
            for fila in spark.table(RESULTADOS).filter(f"id_ejecucion = '{id_ejecucion}'").collect()
        }
        if debe_detener:
            verificar(nombre, "se detiene por una regla de calidad (hay reglas DETENIDO guardadas)", True,
                      any(fila["resultado"] == "DETENIDO" for fila in resultados.values()))
        verificar(nombre, "resultados guardados en hd_dq_resultados (una fila por regla)",
                  reglas_activas(momento), len(resultados))

        for regla, (resultado, incumplen, aisladas) in esperado.items():
            fila = resultados.get(regla)
            verificar(nombre, f"{regla} resultado", resultado, fila["resultado"] if fila else "SIN FILA")
            verificar(nombre, f"{regla} filas que incumplen", incumplen, fila["filas_incumplen"] if fila else "SIN FILA")
            if aisladas is not None:
                verificar(nombre, f"{regla} filas aisladas", aisladas, fila["filas_aisladas"] if fila else "SIN FILA")

        for regla, fila in sorted(resultados.items()):
            if regla not in esperado:
                verificar(nombre, f"{regla} sin efectos colaterales", "CUMPLE", fila["resultado"])

        if not debe_detener:
            estado_log = contar(f"SELECT MAX(estado) FROM {REGISTRO_EJECUCION} WHERE id_ejecucion = '{id_ejecucion}'")
            verificar(nombre, "hd_etl_log cierra EXITOSO", "EXITOSO", estado_log)

        for descripcion, valor_esperado, consulta in extras:
            verificar(nombre, descripcion, valor_esperado, contar(consulta))

    except Exception as error:
        verificar(nombre, "el escenario se ejecuta sin error", "sin error", f"ERROR: {str(error)[:200]}")
    finally:
        restaurar(snapshot)
    print("  ✓ terminó y se deshizo el defecto")

In [0]:
# ============================================================
# Escenarios de control: sin defectos, todo debe dar CUMPLE
# ============================================================

correr_escenario("CONTROL_SILVER", "silver", [], lambda id_ejecucion: ({}, []), debe_detener=False)
correr_escenario("CONTROL_GOLD", "gold", [], lambda id_ejecucion: ({}, []), debe_detener=False)

In [0]:
# ============================================================
# DETENER en Silver: filas duplicadas (DQ-001 a DQ-004)
# ============================================================

CANTIDAD_DUPLICADOS = 3


def inyectar_duplicados(id_ejecucion):
    for tabla, columna in [(PEDIDOS, "order_id"), (DETALLE, "order_item_id"),
                           (CLIENTES, "customer_id"), (PRODUCTOS, "product_card_id")]:
        claves = identificadores(tabla, columna, "estado_calidad = 'VALIDO'", "ASC", CANTIDAD_DUPLICADOS)
        spark.sql(f"INSERT INTO {tabla} SELECT * FROM {tabla} WHERE {columna} IN ({lista(claves)})")
    esperado = {regla: ("DETENIDO", CANTIDAD_DUPLICADOS, None) for regla in ("DQ-001", "DQ-002", "DQ-003", "DQ-004")}
    return esperado, []


correr_escenario("DUPLICADOS_SILVER", "silver", [PEDIDOS, DETALLE, CLIENTES, PRODUCTOS],
                 inyectar_duplicados, debe_detener=True)

In [0]:
# ============================================================
# ADVERTIR en Silver: DQ-007, 010, 017, 018, 019, 020 y 021
# ------------------------------------------------------------
# Cada defecto supera el umbral de su regla. La task no debe detenerse.
# ============================================================

def inyectar_advertencias(id_ejecucion):
    total_pedidos = contar(f"SELECT COUNT(*) FROM {PEDIDOS} WHERE estado_calidad = 'VALIDO'")
    total_lineas = contar(f"SELECT COUNT(*) FROM {DETALLE} WHERE estado_calidad = 'VALIDO'")
    sin_envio = int(total_pedidos * 0.02)       # 2 %: DQ-007 tolera 0,5 %
    envio_antes = int(total_pedidos * 0.015)    # 1,5 %: DQ-017 tolera 0,5 %
    lineas_malas = int(total_lineas * 0.03)     # 3 %: DQ-019, 020 y 021 toleran 1 %

    # DQ-007: pedidos sin fecha de envío (primeras claves)
    claves = identificadores(PEDIDOS, "order_id", "estado_calidad = 'VALIDO'", "ASC", sin_envio)
    spark.sql(f"UPDATE {PEDIDOS} SET shipping_date = NULL WHERE order_id IN ({lista(claves)})")

    # DQ-017: envío 3 días antes del pedido (últimas claves, no se cruzan con las anteriores)
    claves = identificadores(PEDIDOS, "order_id", "estado_calidad = 'VALIDO'", "DESC", envio_antes)
    spark.sql(f"UPDATE {PEDIDOS} SET shipping_date = order_date - INTERVAL 3 DAYS WHERE order_id IN ({lista(claves)})")

    # DQ-010: 5 pedidos con un estado que Gold no conoce (umbral 100 %)
    claves = identificadores(PEDIDOS, "order_id", "estado_calidad = 'VALIDO'", "ASC", 5)
    spark.sql(f"UPDATE {PEDIDOS} SET order_status = 'ESTADO_NUEVO' WHERE order_id IN ({lista(claves)})")

    # DQ-019: precio 0 (y descuento 0 para no cruzarse con DQ-020)
    claves = identificadores(DETALLE, "order_item_id", "estado_calidad = 'VALIDO'", "ASC", lineas_malas)
    spark.sql(f"UPDATE {DETALLE} SET order_item_product_price = 0, order_item_discount = 0 WHERE order_item_id IN ({lista(claves)})")

    # DQ-020 y DQ-021: descuento negativo y tasa mayor que 1 en las mismas líneas (últimas claves)
    claves = identificadores(DETALLE, "order_item_id", "estado_calidad = 'VALIDO'", "DESC", lineas_malas)
    spark.sql(f"UPDATE {DETALLE} SET order_item_discount = -1, order_item_discount_rate = 1.5 WHERE order_item_id IN ({lista(claves)})")

    # DQ-018: la ventana de control apunta a un corte que no existe en Bronze
    spark.sql(f"UPDATE {CONTROL} SET next_start_date = date_add(next_start_date, 400), "
              f"next_end_date = date_add(next_end_date, 400) WHERE pipeline_id = 1")

    esperado = {
        "DQ-007": ("ADVERTENCIA", sin_envio, None),
        "DQ-010": ("ADVERTENCIA", 5, None),
        "DQ-017": ("ADVERTENCIA", envio_antes, None),
        "DQ-018": ("ADVERTENCIA", 1, None),
        "DQ-019": ("ADVERTENCIA", lineas_malas, None),
        "DQ-020": ("ADVERTENCIA", lineas_malas, None),
        "DQ-021": ("ADVERTENCIA", lineas_malas, None),
    }
    return esperado, []


correr_escenario("ADVERTIR_SILVER", "silver", [PEDIDOS, DETALLE, CONTROL],
                 inyectar_advertencias, debe_detener=False)

In [0]:
# ============================================================
# Tolerancia: defectos dentro del umbral no deben dar advertencia
# ============================================================

def inyectar_tolerancia(id_ejecucion):
    total_pedidos = contar(f"SELECT COUNT(*) FROM {PEDIDOS} WHERE estado_calidad = 'VALIDO'")
    total_lineas = contar(f"SELECT COUNT(*) FROM {DETALLE} WHERE estado_calidad = 'VALIDO'")
    sin_envio = int(total_pedidos * 0.001)      # 0,1 %: DQ-007 tolera hasta 0,5 %
    sin_precio = int(total_lineas * 0.005)      # 0,5 %: DQ-019 tolera hasta 1 %
    assert sin_envio > 0 and sin_precio > 0, "No hay datos suficientes para la prueba de tolerancia"

    claves = identificadores(PEDIDOS, "order_id", "estado_calidad = 'VALIDO'", "ASC", sin_envio)
    spark.sql(f"UPDATE {PEDIDOS} SET shipping_date = NULL WHERE order_id IN ({lista(claves)})")

    claves = identificadores(DETALLE, "order_item_id", "estado_calidad = 'VALIDO'", "ASC", sin_precio)
    spark.sql(f"UPDATE {DETALLE} SET order_item_product_price = 0, order_item_discount = 0 WHERE order_item_id IN ({lista(claves)})")

    esperado = {
        "DQ-007": ("CUMPLE", sin_envio, None),
        "DQ-019": ("CUMPLE", sin_precio, None),
    }
    return esperado, []


correr_escenario("TOLERANCIA_SILVER", "silver", [PEDIDOS, DETALLE], inyectar_tolerancia, debe_detener=False)

In [0]:
# ============================================================
# CUARENTENA en Silver: integridad referencial (DQ-011, 012 y 013)
# ------------------------------------------------------------
# Se borra la fila "padre" de la copia de Silver. El motor debe aislar
# las filas huérfanas, registrar el conteo y dejar la task en verde.
# ============================================================

def inyectar_linea_sin_pedido(id_ejecucion):
    pedidos = identificadores(PEDIDOS, "order_id", "estado_calidad = 'VALIDO'", "ASC", 600)
    huerfanas = contar(f"SELECT COUNT(*) FROM {DETALLE} WHERE estado_calidad = 'VALIDO' AND order_id IN ({lista(pedidos)})")
    assert huerfanas > 0, "Los pedidos elegidos no tienen líneas"
    spark.sql(f"DELETE FROM {PEDIDOS} WHERE order_id IN ({lista(pedidos)})")
    esperado = {"DQ-011": ("CUARENTENA", huerfanas, huerfanas)}
    extras = [
        ("líneas en cuarentena con motivo LINEA_SIN_PEDIDO", huerfanas,
         f"SELECT COUNT(*) FROM {DETALLE} WHERE motivo_cuarentena = 'LINEA_SIN_PEDIDO'"),
        ("hd_cuarentena_resumen registra LINEA_SIN_PEDIDO", huerfanas,
         f"SELECT COALESCE(SUM(filas), 0) FROM {CUARENTENA_RESUMEN} "
         f"WHERE id_ejecucion = '{id_ejecucion}' AND motivo_cuarentena = 'LINEA_SIN_PEDIDO'"),
        ("ninguna línea sin pedido sigue como VALIDO", 0,
         f"SELECT COUNT(*) FROM (SELECT order_id FROM {DETALLE} WHERE estado_calidad = 'VALIDO') l "
         f"LEFT ANTI JOIN (SELECT DISTINCT order_id FROM {PEDIDOS}) p ON l.order_id = p.order_id"),
    ]
    return esperado, extras


def inyectar_linea_sin_producto(id_ejecucion):
    productos = identificadores(PRODUCTOS, "product_card_id", "estado_calidad = 'VALIDO'", "ASC", 2)
    huerfanas = contar(f"SELECT COUNT(*) FROM {DETALLE} WHERE estado_calidad = 'VALIDO' "
                       f"AND order_item_cardprod_id IN ({lista(productos)})")
    assert huerfanas > 0, "Los productos elegidos no tienen líneas"
    spark.sql(f"DELETE FROM {PRODUCTOS} WHERE product_card_id IN ({lista(productos)})")
    esperado = {"DQ-012": ("CUARENTENA", huerfanas, huerfanas)}
    extras = [
        ("líneas en cuarentena con motivo LINEA_SIN_PRODUCTO", huerfanas,
         f"SELECT COUNT(*) FROM {DETALLE} WHERE motivo_cuarentena = 'LINEA_SIN_PRODUCTO'"),
        ("hd_cuarentena_resumen registra LINEA_SIN_PRODUCTO", huerfanas,
         f"SELECT COALESCE(SUM(filas), 0) FROM {CUARENTENA_RESUMEN} "
         f"WHERE id_ejecucion = '{id_ejecucion}' AND motivo_cuarentena = 'LINEA_SIN_PRODUCTO'"),
    ]
    return esperado, extras


def inyectar_pedido_sin_cliente(id_ejecucion):
    clientes = identificadores(CLIENTES, "customer_id", "estado_calidad = 'VALIDO'", "ASC", 50)
    huerfanos = contar(f"SELECT COUNT(*) FROM {PEDIDOS} WHERE estado_calidad = 'VALIDO' "
                       f"AND order_customer_id IN ({lista(clientes)})")
    assert huerfanos > 0, "Los clientes elegidos no tienen pedidos"
    spark.sql(f"DELETE FROM {CLIENTES} WHERE customer_id IN ({lista(clientes)})")
    esperado = {"DQ-013": ("CUARENTENA", huerfanos, huerfanos)}
    extras = [
        ("pedidos en cuarentena con motivo PEDIDO_SIN_CLIENTE", huerfanos,
         f"SELECT COUNT(*) FROM {PEDIDOS} WHERE motivo_cuarentena = 'PEDIDO_SIN_CLIENTE'"),
        ("hd_cuarentena_resumen registra PEDIDO_SIN_CLIENTE", huerfanos,
         f"SELECT COALESCE(SUM(filas), 0) FROM {CUARENTENA_RESUMEN} "
         f"WHERE id_ejecucion = '{id_ejecucion}' AND motivo_cuarentena = 'PEDIDO_SIN_CLIENTE'"),
    ]
    return esperado, extras


correr_escenario("CUARENTENA_LINEA_SIN_PEDIDO", "silver", [PEDIDOS, DETALLE], inyectar_linea_sin_pedido, debe_detener=False)
correr_escenario("CUARENTENA_LINEA_SIN_PRODUCTO", "silver", [PRODUCTOS, DETALLE], inyectar_linea_sin_producto, debe_detener=False)
correr_escenario("CUARENTENA_PEDIDO_SIN_CLIENTE", "silver", [CLIENTES, PEDIDOS], inyectar_pedido_sin_cliente, debe_detener=False)

In [0]:
# ============================================================
# DETENER en Gold: DQ-014, DQ-015 y DQ-016
# ============================================================

def inyectar_fallas_gold(id_ejecucion):
    # DQ-014: 3 pedidos vigentes sin la clave de cliente (primeras claves)
    sin_cliente = identificadores(HECHO_PEDIDOS, "order_id", "es_vigente = true", "ASC", 3)
    spark.sql(f"UPDATE {HECHO_PEDIDOS} SET cliente_key = NULL WHERE es_vigente = true AND order_id IN ({lista(sin_cliente)})")

    # DQ-015: 4 pedidos que Silver tiene y Gold no (últimas claves, no se cruzan con las anteriores)
    faltantes = identificadores(HECHO_PEDIDOS, "order_id", "es_vigente = true", "DESC", 4)
    spark.sql(f"DELETE FROM {HECHO_PEDIDOS} WHERE order_id IN ({lista(faltantes)})")

    # DQ-016: 2 pedidos con el resumen duplicado
    repetidos = identificadores(RESUMEN_PEDIDOS, "order_id", "true", "ASC", 2)
    spark.sql(f"INSERT INTO {RESUMEN_PEDIDOS} SELECT * FROM {RESUMEN_PEDIDOS} WHERE order_id IN ({lista(repetidos)})")

    esperado = {
        "DQ-014": ("DETENIDO", 3, None),
        "DQ-015": ("DETENIDO", 4, None),
        "DQ-016": ("DETENIDO", 2, None),
    }
    return esperado, []


correr_escenario("DETENER_GOLD", "gold", [HECHO_PEDIDOS, RESUMEN_PEDIDOS], inyectar_fallas_gold, debe_detener=True)

In [0]:
# ============================================================
# Veredicto final
# ============================================================

df_verificaciones = spark.createDataFrame(
    VERIFICACIONES, "escenario STRING, verificacion STRING, esperado STRING, obtenido STRING, estado STRING"
)
display(df_verificaciones)

total = len(VERIFICACIONES)
fallas = [v for v in VERIFICACIONES if v[4] == "FALLA"]
print(f"Verificaciones: {total} | OK: {total - len(fallas)} | FALLA: {len(fallas)}")

if fallas:
    detalle = "\n- ".join(f"{e} · {d}: esperado {x}, obtenido {o}" for e, d, x, o, _ in fallas)
    raise AssertionError(f"Prueba de reglas de calidad con fallas:\n- {detalle}")

print("✓ PRUEBA DE REGLAS APROBADA: el motor detiene, aísla en cuarentena y advierte según la decisión de cada regla, y respeta la tolerancia del umbral.")